# Models expect a batch of inputs

Let's convert a list of numbers to a tensor and send it to the model:

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = 'distilbert-base-uncased-finetuned-sst-2-english'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = "I've been waiting for a HuggingFace course my whole life."

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)
input_ids = torch.tensor(ids)

# The line below errors out
#model(input_ids)

The problem is that we sent a single sequence to the model, where as Transformers models expect multiple sentences by default. Here we tried to do everything the tokenizer did behind the scenes when we applied it to a `sequence`.

In [2]:
tokenized_inputs = tokenizer(sequence, return_tensors='pt')
print(tokenized_inputs['input_ids'])
print(input_ids)

tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102]])
tensor([ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
         2026,  2878,  2166,  1012])


Try again and add a new dimension:

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = "I've been waiting for a HuggingFace course my whole life."

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)

input_ids = torch.tensor([ids]) #Adding a new dimension
print("Input IDs:", input_ids)

output = model(input_ids)
print("Logits:", output.logits)

Input IDs: tensor([[ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
          2026,  2878,  2166,  1012]])
Logits: tensor([[-2.7276,  2.8789]], grad_fn=<AddmmBackward0>)


*Batching* is the act of sending multiple sentences through the model, all at once. If you only have one sentence, you can just build a batch with a single sequence

In [4]:
batched_ids = [ids, ids] # This is a batch of two identical sequences

### Example: Converting batched ids into a tensor and passing it through a model

In [5]:
batched_input_ids = torch.tensor(batched_ids)

print(f"Batched Input IDs: {batched_input_ids}")

batched_output = model(batched_input_ids)
print("Logits:", batched_output.logits)

Batched Input IDs: tensor([[ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
          2026,  2878,  2166,  1012],
        [ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
          2026,  2878,  2166,  1012]])
Logits: tensor([[-2.7276,  2.8789],
        [-2.7276,  2.8789]], grad_fn=<AddmmBackward0>)


Batching allows the model to work when you feed it multiple sentences. Using multiple sequences is just as simple as building a batch with a single sequence.

There is a second issue, though. When you're trying to batch together two (or more) sentences, they might be of different lengths. If you've ever worked with tensors before, you know that they need to be of rectangular shape, so you won't be able to convert the list of input IDs into a tensor directly. 

To work around this problem, we usually `pad` the inputs

# Padding the inputs

The following list of lists cannot be converted to a tensor:

In [6]:
batched_ids = [
    [200,200, 200],
    [200, 200]
]

In order to work around this, we'll use `padding` to make our tensors have rectangular shape. 

Padding makes sure all our sentences have the same legnth by adding a special word called the `padding token` to the sentences with fewer values.

For example, if you have 10 sentences with 10 words and 1 sentence with 20 words, padding will ensure al the sentences have 20 words. In our example, the result tensor looks like this:

In [7]:
padding_id = 100

batched_ids = [
    [200,200,200],
    [200,200, padding_id],
]

The padding token ID can be found in `tokenizer.pad_token_id`. Let's use it and send our two sentences through the model individually and batched together: 

In [8]:
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence1_ids = [[200, 200, 200]]
sequence2_ids = [[200, 200]]
batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],
]

print(model(torch.tensor(sequence1_ids)).logits)
print(model(torch.tensor(sequence2_ids)).logits)
print(model(torch.tensor(batched_ids)).logits)

We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


tensor([[ 1.5694, -1.3895]], grad_fn=<AddmmBackward0>)
tensor([[ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)
tensor([[ 1.5694, -1.3895],
        [ 1.3374, -1.2163]], grad_fn=<AddmmBackward0>)


There's something wrong with the logits in our batched predictions: the second row should be the same as the lgoits for the second sentence, but we've got completely different values!

This is because the key feature of Transformer models is attention layers that `contextualize` each token. These will take into account the padding tokens since they attend to all of the tokens of a sequence.

To get the same result when passing individual sentences of different lengths through the model or when passing a batch with the same sentences and padding applied, we need to tell those attention layers to ignore the padding tokens. This is done by using an attention mask.

# Attention masks

`Attention masks` are tensors with the exact same shape as the input IDs tensor, filled with 0s and 1s: 

1s indicate the corresponding tokens should be attended to, 

0s indicate the corresponding tokens should not be attended to (i.e., they should be ignored by the attention layers of the model).

In [9]:
batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],
]

attention_mask = [
    [1,1,1],
    [1,1,0],
]

outputs = model(torch.tensor(batched_ids), attention_mask=torch.tensor(attention_mask))
print(outputs.logits)


tensor([[ 1.5694, -1.3895],
        [ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)


Now we get the same lgots for the second sentence in the batch.

Notice how the last value of the second sequence is a padding ID, which is a 0 value in the attention mask.

### Example: Manually converting two more sentences with a tokenizer and comparing them

In [16]:
checkpoint = 'distilbert-base-uncased-finetuned-sst-2-english'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence_1 = "I've been waiting for a HuggingFace course my whole life."
sequence_2 = "I hate this so much!"

# Tokenize
tokens_1 = tokenizer.tokenize(sequence_1)
tokens_2 = tokenizer.tokenize(sequence_2)

# Convert token to ids
ids_1 = tokenizer.convert_tokens_to_ids(tokens_1)
ids_2 = tokenizer.convert_tokens_to_ids(tokens_2)

# Convert ids to tensor so we can feed them to the model
input_ids_1 = torch.tensor([ids_1])
input_ids_2 = torch.tensor([ids_2])

# Feed to Model
output_1 = model(input_ids_1)
output_2 = model(input_ids_2)

# Batching to compare output results

# Adding padding to the ids 
while len(ids_2) < len(ids_1):
    ids_2.append(tokenizer.pad_token_id)

batched_ids = [
    ids_1,
    ids_2
]

# Create attention masks for all sequences [So the model knowns which tokens to attend to, and which ones NOT to attend to
attention_mask_1 = []
attention_mask_2 = []

for tok_id in ids_1:
    if tok_id != tokenizer.pad_token_id:
        attention_mask_1.append(1)
    else:
        attention_mask_1.append(0)

for tok_id in ids_2:
    if tok_id != tokenizer.pad_token_id:
        attention_mask_2.append(1)
    else:
        attention_mask_2.append(0)

attention_mask = [attention_mask_1, attention_mask_2]

batched_input_ids = torch.tensor(batched_ids)

output_3 = model(batched_input_ids, attention_mask=torch.tensor(attention_mask))

print(output_1.logits)
print(output_2.logits)
print(output_3.logits)


tensor([[-2.7276,  2.8789]], grad_fn=<AddmmBackward0>)
tensor([[ 3.1931, -2.6685]], grad_fn=<AddmmBackward0>)
tensor([[-2.7276,  2.8789],
        [ 3.1931, -2.6685]], grad_fn=<AddmmBackward0>)


# Longer sequences

With Transformer models, there is a limit to the lengths of the sequences we can pass to the models.

Most models handle sequences of up to 512 or 1024 tokens, and will crash when asked to process longer sequences.

There are two solutions to this problem:
* Use a model with a longer supported sequence length.
* Truncate your sequences.

Models have different supported sequence lengths, and some specialize in handling very long sequences. LongFormer is one example, and another is LED. If you are working on a task that requires very long sequences, HuggingFace recommends you take a look at these models.

Otherwise, they recommend you truncate your sequences by specifying the `max_sequence_length` parameter:

`sequence = sequence[:max_sequence_length]`